#  Causal Transformer based models using pretrained word embeddings and position embeddings with relative positions

In [51]:
# === FULL UPDATED CAUSAL TRANSFORMER CHATBOT ===

!pip install -q nltk evaluate transformers sentencepiece datasets

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import evaluate
import nltk
nltk.download('punkt')

# === LOAD DATA ===
from ast import literal_eval
passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
queries = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")
passages = passages[passages['passage'].apply(lambda x: isinstance(x, str) and x.strip() != "")].copy()
passage_dict = dict(zip(passages.index, passages['passage']))
queries['relevant_passage_ids'] = queries['relevant_passage_ids'].apply(literal_eval)
def get_context(pids): return " ".join([passage_dict.get(pid, "") for pid in pids if pid in passage_dict])
queries['context'] = queries['relevant_passage_ids'].apply(get_context)
queries['prompt'] = queries.apply(lambda row: f"Question: {row['question']}\nContext: {row['context']}", axis=1)
queries['response'] = queries['answer']
final_df = queries[['prompt', 'response']].dropna()

# === SPLIT ===
train_df, test_df = train_test_split(final_df, test_size=0.1, random_state=42)

# === TOKENIZER ===
from transformers import PreTrainedTokenizerFast
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
texts = (train_df['prompt'].tolist() + train_df['response'].tolist())
tokenizer_model = Tokenizer(models.BPE())
tokenizer_model.pre_tokenizer = pre_tokenizers.Whitespace()
trainer = trainers.BpeTrainer(special_tokens=["<PAD>", "<UNK>", "<BOS>", "<EOS>", "<SEP>"])
tokenizer_model.train_from_iterator(texts, trainer)
tokenizer = PreTrainedTokenizerFast(tokenizer_object=tokenizer_model)
tokenizer.add_special_tokens({'pad_token': '<PAD>', 'unk_token': '<UNK>', 'bos_token': '<BOS>', 'eos_token': '<EOS>', 'sep_token': '<SEP>'})

# === MODEL COMPONENTS ===
vocab_size = tokenizer.vocab_size
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RelativePositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.rel_pos = nn.Parameter(torch.randn(max_len, d_model))
    def forward(self, seq_len):
        return self.rel_pos[:seq_len, :]

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B, T, C = x.size()
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.d_head).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        scores = (q @ k.transpose(-2, -1)) / np.sqrt(self.d_head)
        mask = torch.tril(torch.ones(T, T, device=x.device)).unsqueeze(0).unsqueeze(0)
        scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        return self.out(out)

class CausalTransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model=512, n_heads=8, n_layers=6, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = RelativePositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                'attn': CausalSelfAttention(d_model, n_heads),
                'norm1': nn.LayerNorm(d_model),
                'ff': nn.Sequential(nn.Linear(d_model, 4*d_model), nn.ReLU(), nn.Linear(4*d_model, d_model)),
                'norm2': nn.LayerNorm(d_model)
            }) for _ in range(n_layers)
        ])
        self.out_proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        T = x.size(1)
        x = self.embedding(x) + self.pos_encoding(T)
        for layer in self.layers:
            attn_out = layer['attn'](x)
            x = layer['norm1'](x + attn_out)
            ff_out = layer['ff'](x)
            x = layer['norm2'](x + ff_out)
        return self.out_proj(x)

# === TOP-K SAMPLING FUNCTION ===
def top_k_top_p_filtering(logits, top_k=0, top_p=0.0, filter_value=-float('Inf')):
    if top_k > 0:
        top_k = min(top_k, logits.size(-1))
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = filter_value
    # Top-p (nucleus) filtering can be added here if needed
    return logits

# === TRAINING SETUP ===
model = CausalTransformerModel(vocab_size).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

max_len = 256
batch_size = 32

def encode_row(row):
    prompt_ids = tokenizer.encode("<BOS>" + row['prompt'] + "<SEP>")
    response_ids = tokenizer.encode(row['response'] + "<EOS>")
    full_input = (prompt_ids + response_ids)[:max_len]
    input_ids = full_input[:-1] + [tokenizer.pad_token_id] * (max_len - len(full_input[:-1]))
    target_ids = full_input[1:] + [tokenizer.pad_token_id] * (max_len - len(full_input[1:]))
    return torch.tensor(input_ids), torch.tensor(target_ids)

train_batches = [train_df.iloc[i:i+batch_size] for i in range(0, len(train_df), batch_size)]

# === TRAIN LOOP ===
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_batches):
        inputs, targets = zip(*[encode_row(row) for _, row in batch.iterrows()])
        inputs = torch.stack(inputs).to(device)
        targets = torch.stack(targets).to(device)
        logits = model(inputs)
        loss = criterion(logits.view(-1, vocab_size), targets.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_batches):.4f}")

# === GENERATION WITH TOP-K SAMPLING ===
def generate(prompt, max_len=50, top_k=50, temperature=0.8):
    model.eval()
    input_ids = tokenizer.encode("<BOS>" + prompt + "<SEP>")[:max_len]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    generated = input_tensor
    with torch.no_grad():
        for _ in range(max_len):
            logits = model(generated)[:, -1, :]
            filtered_logits = top_k_top_p_filtering(logits, top_k=top_k)
            probs = F.softmax(filtered_logits / temperature, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated = torch.cat([generated, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0].tolist(), skip_special_tokens=True)

# === EVALUATION ON FULL TEST SET ===
preds, refs = [], []
for idx in tqdm(range(len(test_df))):
    row = test_df.iloc[idx]
    preds.append(generate(row['prompt']))
    refs.append(row['response'])

rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")
rouge_result = rouge.compute(predictions=preds, references=refs)
bert_result = bertscore.compute(predictions=preds, references=refs, lang="en")
print(f"\nROUGE-L: {rouge_result['rougeL']:.4f}")
print(f"BERTScore F1: {np.mean(bert_result['f1']):.4f}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
100%|██████████| 133/133 [01:37<00:00,  1.37it/s]


Epoch 1, Loss: 7.0956


100%|██████████| 133/133 [01:39<00:00,  1.34it/s]


Epoch 2, Loss: 6.0828


100%|██████████| 133/133 [01:38<00:00,  1.34it/s]


...

Epoch 9, Loss: 4.2649


100%|██████████| 133/133 [01:38<00:00,  1.35it/s]


Epoch 10, Loss: 4.2060


100%|██████████| 472/472 [02:07<00:00,  3.70it/s]
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



ROUGE-L: 0.1411
BERTScore F1: 0.8214
